In [1]:
import json
import boto3
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
endpoint = os.getenv("S3_ENDPOINT", os.getenv("SEAWEEDFS_S3_ENDPOINT", "http://127.0.0.1:8333"))
if endpoint and not str(endpoint).startswith("http"):
    endpoint = f"http://{endpoint}"

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID", "admin"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY", "admin"),
    region_name=os.getenv("AWS_REGION", "us-east-1"),
)
print("S3 endpoint:", endpoint)

S3 endpoint: http://seaweedfs:8333


**JSON**

In [14]:
# Coger el JSON de esta "carpeta" y imprimirlo (si hay varios, usa el primero; cambia a keys[-1] si quieres el último)
bucket = "landing-zone"
prefix = "transit_posts"  # misma ruta que listas arriba; acaba en /

keys = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        k = obj["Key"]
        if k.endswith(".json"):
            keys.append(k)

if not keys:
    raise FileNotFoundError(f"No hay ningún .json bajo s3://{bucket}/{prefix}")

key = keys[0]
print("Leyendo:", key, "\n")

raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
data = json.loads(raw.decode("utf-8"))
print(json.dumps(data, indent=2, ensure_ascii=False))

Leyendo: transit_posts/post_2026-05-05T18:37:45+00:00.json 

{
  "title": "RT by @transit: 🏍️ Per reforçar la seguretat dels motoristes aquesta setmana farem amb policies locals i @transit 715 controls estàtics i dinàmics amb 2.285 agents.\n\nI utilitzarem motocicletes espiell amb l’objectiu de detectar infraccions que els posin en perill 👇🏼\n\nhttps://x.com/transit/status/2051261138701082864?s=20",
  "title_detail": {
    "type": "text/plain",
    "language": null,
    "base": "https://nitter.net/transit/rss",
    "value": "RT by @transit: 🏍️ Per reforçar la seguretat dels motoristes aquesta setmana farem amb policies locals i @transit 715 controls estàtics i dinàmics amb 2.285 agents.\n\nI utilitzarem motocicletes espiell amb l’objectiu de detectar infraccions que els posin en perill 👇🏼\n\nhttps://x.com/transit/status/2051261138701082864?s=20"
  },
  "authors": [
    {
      "name": "@mossos"
    }
  ],
  "author": "@mossos",
  "author_detail": {
    "name": "@mossos"
  },
  "summary

**CSV**

In [23]:
import pandas as pd
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "landing-zone"
prefix = "ajuntament_bcn/trams_congestio"

# Llista tots els CSVs
csvs = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".csv"):
            csvs.append(obj["Key"])

print(f"CSVs trobats: {len(csvs)}")

# Llegeix tots i ajunta'ls en un sol DataFrame
dfs = []
for key in csvs:
    raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
    df = pd.read_csv(io.BytesIO(raw))
    print(f"  → {key}: {len(df)} files -> columns: {df.columns} -> numero de nulos: {df.isna().sum()}")
    dfs.append(df)

# Ajunta tots
df_total = pd.concat(dfs, ignore_index=True)
print(f"\nTotal files combinades: {len(df_total)}")
print(df_total)


CSVs trobats: 17
  → ajuntament_bcn/trams_congestion/2025_01_Gener_TRAMS_TRAMS.csv: 1970528 files -> columns: Index(['idTram', 'data', 'estatActual', 'estatPrevist'], dtype='str') -> numero de nulos: idTram          0
data            0
estatActual     0
estatPrevist    0
dtype: int64
  → ajuntament_bcn/trams_congestion/2025_02_Febrer_TRAMS_TRAMS.csv: 1786456 files -> columns: Index(['idTram', 'data', 'estatActual', 'estatPrevist'], dtype='str') -> numero de nulos: idTram          0
data            0
estatActual     0
estatPrevist    0
dtype: int64
  → ajuntament_bcn/trams_congestion/2025_03_Marc_TRAMS_TRAMS.csv: 2207800 files -> columns: Index(['idTram', 'data', 'estatActual', 'estatPrevist'], dtype='str') -> numero de nulos: idTram          0
data            0
estatActual     0
estatPrevist    0
dtype: int64
  → ajuntament_bcn/trams_congestion/2025_04_Abril_TRAMS_TRAMS.csv: 1542800 files -> columns: Index(['idTram', 'data', 'estatActual', 'estatPrevist'], dtype='str') -> numero de nul

**XML**

In [30]:
import boto3

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "landing-zone"
prefix = "sct/traffic_incidents/"

# Llista tots els XMLs
xmls = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".xml"):
            xmls.append(obj["Key"])

print(f"XMLs trobats: {len(xmls)}")

# Llegeix i imprimeix cada un
for key in xmls:
    print(f"\n{'='*50}")
    print(f"Fitxer: {key}")
    print('='*50)
    raw = client.get_object(Bucket=bucket, Key=key)["Body"].read()
    print(raw.decode("utf-8"))

XMLs trobats: 1

Fitxer: sct/traffic_incidents/2026-05-06T23:25:00+00:00.xml
<?xml version="1.0" encoding="UTF-8"?><wfs:FeatureCollection xmlns="http://www.opengis.net/wfs" xmlns:wfs="http://www.opengis.net/wfs" xmlns:gml="http://www.opengis.net/gml" xmlns:cite="http://www.opengeospatial.net/cite" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengeospatial.net/cite http://localhost:8080/sct-gis/wfs?service=WFS&amp;version=1.0.0&amp;request=DescribeFeatureType&amp;typeName=cite%3Amct2_v_afectacions_data http://www.opengis.net/wfs http://localhost:8080/sct-gis/schemas/wfs/1.0.0/WFS-basic.xsd"><gml:boundedBy><gml:null>unknown</gml:null></gml:boundedBy><gml:featureMember><cite:mct2_v_afectacions_data fid="mct2_v_afectacions_data.fid-1f259c63_19dfd8abf28_6ac9"><cite:geom><gml:Point srsName="http://www.opengis.net/gml/srs/epsg.xml#4326"><gml:coordinates xmlns:gml="http://www.opengis.net/gml" decimal="." cs="," ts=" ">1.98161463,41.4605575</gml:coordina

**PERSISTENT-LANDING-ZONE**

In [31]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "persistent-landing-zone"
prefix = "sct/traffic_incidents/"


parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.dtypes)
    

Parquets trobats: 1
Total files: 1
Index(['file_name', 'xml_content'], dtype='str')
                                           file_name  \
0  sct/traffic_incidents/2026-05-06T23:25:00+00:0...   

                                         xml_content  
0  <?xml version="1.0" encoding="UTF-8"?><wfs:Fea...  
file_name      str
xml_content    str
dtype: object


**TRUSTED ZONE**

In [31]:
import pyarrow.parquet as pq
import pyarrow as pa
import boto3
import io

client = boto3.client(
    "s3",
    endpoint_url="http://localhost:7333",
    aws_access_key_id="admin",
    aws_secret_access_key="admin",
    region_name="us-east-1",
)

bucket = "trusted-zone"
prefix = "ajuntament_bcn/trams_congestion/"


parquets = []
for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"Parquets trobats: {len(parquets)}")

if parquets:
    raw = client.get_object(Bucket=bucket, Key=parquets[0])["Body"].read()
    
    # Llegeix ignorant els tipus d'extensió de pandas
    reader = pq.ParquetFile(io.BytesIO(raw))
    table = reader.read()
    
    # Converteix a pandas ignorant metadades problemàtiques
    df = table.to_pandas(timestamp_as_object=True)
    print(f"Total files: {len(df)}")
    print(df.columns)
    print(df.head())
    print(df.dtypes)

    

Parquets trobats: 1
Total files: 1064
Index(['idTram', 'data', 'estatActual', 'estatPrevist'], dtype='str')
   idTram          data  estatActual  estatPrevist
0    54.0  2.026061e+13          0.0           0.0
1    64.0  2.026061e+13          0.0           0.0
2    88.0  2.026061e+13          1.0           1.0
3   126.0  2.026061e+13          0.0           0.0
4   136.0  2.026061e+13          0.0           0.0
idTram          float64
data            float64
estatActual     float64
estatPrevist    float64
dtype: object


In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa
from IPython.display import display

import boto3
import io
import pandas as pd

# ─── CONFIGURACIÓ ─────────────────────────────────────────────────────────────
ENDPOINT    = "http://localhost:7333"
ACCESS_KEY  = "admin"
SECRET_KEY  = "admin"
BUCKET      = "exploitation-zone"
PREFIX      = "incidents/daily_incidents/"   # <-- canvia aquí
# ──────────────────────────────────────────────────────────────────────────────

client = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name="us-east-1",
)

# 1. Llistar tots els fitxers .parquet del prefix
parquets = []
paginator = client.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            parquets.append(obj["Key"])

print(f"📦 Parquets trobats: {len(parquets)}")
for p in parquets:
    print(f"   - {p}")

if not parquets:
    print("⚠️  No s'han trobat fitxers .parquet")
else:
    # 2. Llegir tots i concatenar
    dfs = []
    for key in parquets:
        raw = client.get_object(Bucket=BUCKET, Key=key)["Body"].read()
        table = pq.ParquetFile(io.BytesIO(raw)).read()
        dfs.append(table.to_pandas(timestamp_as_object=True))
        print(f"   ✅ Llegit: {key}  →  {len(dfs[-1])} files")

    df = pd.concat(dfs, ignore_index=True)

    # 3. Resum
    print(f"\n📊 Total files : {len(df)}")
    print(f"   Columnes    : {list(df.columns)}")
    print(f"\n{df.dtypes}")
    print(f"\n{df.head()}")
    print(f"\n{df.iloc[0:20]}")



📦 Parquets trobats: 2
   - incidents/daily_incidents/part-00000-e939b25a-b530-47ee-bb95-57cdd6ec6e4c-c000.snappy.parquet
   - incidents/daily_incidents/part-00001-b712349e-1eaa-48ce-8981-5af00c6556f4-c000.snappy.parquet
   ✅ Llegit: incidents/daily_incidents/part-00000-e939b25a-b530-47ee-bb95-57cdd6ec6e4c-c000.snappy.parquet  →  0 files
   ✅ Llegit: incidents/daily_incidents/part-00001-b712349e-1eaa-48ce-8981-5af00c6556f4-c000.snappy.parquet  →  3 files

📊 Total files : 3
   Columnes    : ['font', 'data_ts', 'titol', 'descripcio', 'carretera', 'pk_inici', 'pk_fi', 'autor', 'link']

font          str
data_ts       str
titol         str
descripcio    str
carretera     str
pk_inici      str
pk_fi         str
autor         str
link          str
dtype: object

           font     data_ts  \
0  transit_post  2026-06-06   
1  transit_post  2026-06-06   
2  transit_post  2026-06-06   

                                               titol  \
0  ACCIDENT. Calçada tallada (Retenció): AP-7 | V... 